# Tropical Cyclones with Neighborhood Filters

A neighborhood reduction replaces the value at every grid element with a reduction
(mean, min, max, sum, ...) over all elements within a great-circle radius `r` of it.
That is a *local* operation, and several standard tropical-cyclone diagnostics are
built from exactly that:

| Diagnostic | Neighborhood expression |
|---|---|
| Storm center | cells where `PS == nb.min(PS)` |
| Pressure deficit | `nb.mean(PS) - PS` |
| Warm-core anomaly | `T - nb.mean(T)` |
| Integrated kinetic energy (IKE) | `nb.sum(ke * area)`, read at the center |
| Peak wind near the center | `nb.max(wind_speed)`, read at the center |
| Wind footprint | `nb.max(on_track) > 0` dilates the track by `r` |

**Part 1** finds and characterizes a real cyclone in a model snapshot.
**Part 2** follows a storm through its life cycle and computes IKE, a wind-field
alternative to ACE. The tutorial data has no winds or time series, so Part 2 uses a
synthetic storm on the same grid.

For an introduction to the API itself, see the
[Neighborhood Filter](neighborhood-filter.ipynb) user guide.

In [ ]:
import holoviews as hv
import numpy as np
import pandas as pd
import xarray as xr

import uxarray as ux

# Load the plotting extension up front; some frontends (e.g. VS Code) leave the
# first plot blank if it is loaded in the same cell as the plot.
hv.extension("bokeh")

R_EARTH = 6.371e6  # m


def at(field, n_face):
    # Value of a grid-mapped field at the given face(s). Converting to a plain
    # DataArray lets an index that varies along `time` select pointwise.
    return xr.DataArray(field).isel(n_face=n_face)

## Part 1: A Cyclone in a Model Snapshot

`ne120-tcsubset` is a regional subset of a ~25 km cubed-sphere simulation with a
tropical cyclone off northwest Australia: surface pressure plus temperature and
geopotential height on pressure levels.

In [ ]:
ds = ux.tutorial.open_dataset("ne120-tcsubset")
grid = ds.uxgrid
lat, lon = grid.face_lat.values, grid.face_lon.values
ds

In [ ]:
ps_plot = (ds["PS"] / 100).plot.polygons(
    cmap="viridis", clabel="PS (hPa)", title="Surface pressure", width=500, height=450
)
ps_plot

### Build the neighborhood once

The expensive part of a neighborhood reduction is finding the neighbors. Building it
from the grid with `Grid.neighborhood` does that once, after which any number of
variables and reductions reuse it.

This is a regional grid, so neighborhoods near the boundary are cut off and contain
fewer cells. `n_neighbors` shows where: a minimum or mean computed there only sees one
side of the storm, so we mask those cells out.

In [ ]:
nb_1deg = grid.neighborhood(r=1.0)
nb_1deg

In [ ]:
counts = nb_1deg.n_neighbors
interior = counts >= 0.8 * counts.median()

ux.UxDataArray(counts, uxgrid=grid).plot.polygons(
    cmap="cividis",
    title="Cells within 1° (edge cells are truncated)",
    width=500,
    height=450,
)

### Find the center

A cell is a local minimum if it equals the minimum of its own neighborhood. This is
the same test storm trackers such as TempestExtremes start from.

In [ ]:
ps = ds["PS"]
is_min = (ps == nb_1deg.min(ps)) & interior

center = int(np.flatnonzero(is_min.values)[0])
print(f"local minima away from the edge: {int(is_min.sum())}")
print(
    f"center: {-lat[center]:.2f}°S, {lon[center]:.2f}°E, PS = {ps.values[center] / 100:.1f} hPa"
)

ps_plot * hv.Points([(lon[center], lat[center])]).opts(color="red", marker="x", size=14)

### How deep is it?

The pressure deficit is the central pressure relative to its surroundings, and
"surroundings" needs a scale. Measured against larger neighborhoods, the same low
looks deeper, so a tracking threshold always comes with a radius.

In [ ]:
for r in [0.5, 1.0, 2.0]:
    depth = grid.neighborhood(r=r).mean(ps) - ps
    print(f"r = {r}°: deficit {depth.values[center] / 100:5.1f} hPa")

### Is it warm-core?

Tropical cyclones have a warm core aloft. Subtracting each level's 2° neighborhood
mean removes the large-scale environment and leaves the local anomaly. The
neighborhood is reused across all 26 pressure levels, which pass through as an
extra dimension.

In [ ]:
nb_2deg = grid.neighborhood(r=2.0)
t_anom = ds["T"] - nb_2deg.mean(ds["T"])

profile = at(t_anom, center).sel(plev=slice(10000, None))  # surface to 100 hPa
profile_plot = hv.Curve(
    (profile.values, profile.plev.values / 100), "T anomaly (K)", "Pressure (hPa)"
).opts(invert_yaxis=True, width=300, height=450, title="Above the center")

map_400 = t_anom.sel(plev=40000, method="nearest").plot.polygons(
    cmap="RdBu_r",
    symmetric=True,
    clabel="K",
    title="T anomaly near 400 hPa",
    width=500,
    height=450,
)
profile_plot + map_400

The anomaly peaks near 370 hPa, about 8 K warmer than its surroundings, above the
surface center: a classic tropical-cyclone warm core. Together,
the local minimum, the pressure deficit, and the warm core are the usual criteria
for telling a tropical cyclone apart from other lows.

## Part 2: Following a Storm's Life Cycle

ACE (accumulated cyclone energy) sums the square of one number, the peak wind, every
six hours. It cannot tell a compact storm from a broad one with the same peak wind,
even though the broad one moves far more energy and drives a larger storm surge.
**Integrated kinetic energy** (IKE; Powell and Reinhold 2007) integrates the kinetic
energy of all tropical-storm-force wind (≥ 18 m/s) over the area around the center:

$$\mathrm{IKE} = \int_A \tfrac{1}{2} \rho\, |\mathbf{U}|^2 \, dA \quad (\text{1 m deep layer},\ \rho = 1\ \mathrm{kg\,m^{-3}})$$

An integral over a disk around the center is exactly an area-weighted neighborhood
**sum** evaluated at the center.

### A synthetic storm

The tutorial data has no winds and no time series, so we build a storm on the same
grid from the Holland (1980) wind profile, which gives physically consistent
pressure and wind fields. Over 48 hourly steps the storm moves southwest, intensifies
until hour 24, then weakens while its wind field broadens, as storms often do late in
life. A weak, stationary low that is not a cyclone is added to give the tracker
something to reject.

In [ ]:
hours = np.arange(49)
time = np.datetime64("2026-01-10T00") + hours.astype("timedelta64[h]")

# Track and life cycle
center_lat = np.linspace(-17.0, -19.0, hours.size)
center_lon = np.linspace(115.5, 112.5, hours.size)
deficit = np.interp(hours, [0, 24, 48], [1500, 5500, 3500])  # Pa
rmax = np.interp(hours, [0, 24, 48], [25e3, 40e3, 70e3])  # radius of max wind, m
shape_b = np.interp(hours, [0, 24, 48], [1.6, 1.6, 1.3])  # Holland B; lower is broader


def holland_storm(t, rho=1.15):
    # Surface pressure and wind components of a Holland (1980) vortex at hour t.
    phi, lam = np.radians(lat), np.radians(lon)
    phi_c, lam_c = np.radians(center_lat[t]), np.radians(center_lon[t])
    hav = (
        np.sin((phi - phi_c) / 2) ** 2
        + np.cos(phi) * np.cos(phi_c) * np.sin((lam - lam_c) / 2) ** 2
    )
    r = 2 * R_EARTH * np.arcsin(np.sqrt(hav)) + 1.0  # +1 m: no division by zero
    x = (rmax[t] / r) ** shape_b[t]
    f = abs(2 * 7.292e-5 * np.sin(phi_c))
    speed = np.sqrt(shape_b[t] / rho * deficit[t] * x * np.exp(-x) + (r * f / 2) ** 2)
    speed -= r * f / 2

    environment = 101000 - 60 * (lat + 18)  # weak north-south gradient, Pa
    pressure = environment - deficit[t] * (1 - np.exp(-x))

    # Clockwise rotation: cyclonic in the Southern Hemisphere
    dx = (lam - lam_c) * np.cos(phi_c) * R_EARTH
    dy = (phi - phi_c) * R_EARTH
    return pressure, speed * dy / r, -speed * dx / r


pressure, u, v = map(np.stack, zip(*(holland_storm(t) for t in hours)))
pressure -= 400 * np.exp(-(((lat + 15) ** 2 + ((lon - 111.5) * 0.96) ** 2) / 2.5**2))


def as_field(values, name, units):
    return ux.UxDataArray(
        values,
        dims=["time", "n_face"],
        coords={"time": time},
        uxgrid=grid,
        name=name,
        attrs={"units": units},
    )


storm_ps = as_field(pressure, "PS", "Pa")
wspd = np.hypot(as_field(u, "U", "m/s"), as_field(v, "V", "m/s")).rename("wind_speed")
wspd

### Track the center

The same test as in Part 1, now applied to all 49 time steps in a single call. The
synthetic fields live on the same grid, so `nb_1deg` and `interior` from Part 1 are
reused as they are. A minimum pressure deficit of 2 hPa separates the cyclone from
the weak low.

In [ ]:
is_min = storm_ps == nb_1deg.min(storm_ps)
depth = nb_1deg.mean(storm_ps) - storm_ps
candidates = is_min & interior & (depth > 200)

print("local minima per hour:     ", np.unique(is_min.sum("n_face")))
print("cyclone candidates per hour:", np.unique(candidates.sum("n_face")))

In [ ]:
# Link candidates into a track: start at the deepest, then step to the nearest
# candidate each hour. Real output would also need to handle gaps and several storms.
track = np.empty(hours.size, dtype=int)
for t in hours:
    found = np.flatnonzero(candidates.values[t])
    if t == 0:
        track[t] = found[np.argmin(storm_ps.values[t, found])]
    else:
        prev = track[t - 1]
        jump = np.hypot(
            lat[found] - lat[prev],
            (lon[found] - lon[prev]) * np.cos(np.radians(lat[prev])),
        )
        track[t] = found[np.argmin(jump)]

track = xr.DataArray(track, dims="time", coords={"time": time})
error = np.hypot(lat[track] - center_lat, lon[track] - center_lon)
print(
    f"largest distance from the true center: {error.max():.2f}° (grid spacing ~0.25°)"
)

### IKE, peak wind, and storm size

One 3° (~330 km) neighborhood serves three diagnostics at every hour. Its radius
bounds the integral so that other weather systems are not counted; it must be large
enough to contain the storm's gale-force winds, which it is here.

In [ ]:
area = grid.face_areas * R_EARTH**2  # m²; face_areas is on the unit sphere
gale = wspd >= 18  # tropical-storm force, m/s
ke = (0.5 * 1.0 * wspd**2).where(gale, 0) * area  # J per 1 m deep layer

nb_storm = grid.neighborhood(r=3.0)

ike = at(nb_storm.sum(ke), track) / 1e12  # TJ
vmax = at(nb_storm.max(wspd), track)  # m/s
gale_radius = np.sqrt(at(nb_storm.sum(gale * area), track) / np.pi) / 1e3  # km

curve = dict(width=300, height=250, tools=["hover"])
(
    hv.Curve((time, vmax.values), "time", "Peak wind (m/s)").opts(**curve)
    + hv.Curve((time, ike.values), "time", "IKE (TJ)").opts(**curve)
    + hv.Curve((time, gale_radius.values), "time", "Gale-force radius (km)").opts(
        **curve
    )
)

In [ ]:
fixes = time[::6]  # the 6-hourly fixes ACE is built from
vmax_kt = vmax.sel(time=fixes) * 1.94384
pd.DataFrame(
    {
        "peak wind (kt)": vmax_kt.round(0),
        "ACE term (1e4 kt²)": (1e-4 * vmax_kt.where(vmax_kt >= 35, 0) ** 2).round(2),
        "IKE (TJ)": ike.sel(time=fixes).round(1),
    },
    index=pd.Index(fixes, name="time"),
)

The peak wind tops out at hour 24 and falls 30% by hour 48, halving each ACE term.
Over the same period the gale-force wind field keeps expanding and IKE keeps rising,
peaking 18 hours after the peak wind. A diagnostic built on one wind value cannot see
this; one built on the whole wind field can.

### Wind footprint

Which areas felt damaging wind, and how strong? Marking the center at each hour and
taking a neighborhood **max** of that marker dilates it to everything within 3° of
the center. Masking the wind with it before taking the maximum over time keeps other
weather systems out of the footprint. Here the synthetic field has only one storm, so
the mask changes nothing, but on real output it does.

In [ ]:
marker = np.zeros(wspd.shape)
marker[hours, track.values] = 1
near_track = nb_storm.max(as_field(marker, "on_track", "1")) > 0

footprint = wspd.where(near_track).max("time").rename("max_wind")
footprint.plot.polygons(
    cmap="inferno",
    clabel="Max wind (m/s)",
    title="48 h wind footprint",
    width=500,
    height=450,
) * hv.Path([np.column_stack([lon[track], lat[track]])]).opts(
    color="white", line_width=2
)

### Larger output

On a long, high-resolution run, chunk along `time`. A neighborhood can span the
whole grid, so `n_face` must stay in one chunk (uxarray collapses it with a warning
if it is chunked). The neighbor search is still done only once, and the time chunks
are reduced in parallel.

In [ ]:
lazy_vmax = nb_storm.max(wspd.chunk(time=12))
print(type(lazy_vmax.data).__name__, lazy_vmax.data.numblocks)
print(
    "matches the in-memory result:",
    np.allclose(lazy_vmax.values, nb_storm.max(wspd).values),
)

## Part 3: The 2020 Atlantic Season in ERA5

The record 2020 season had 30 named storms. Here the Part 2 pipeline runs on every
hour of it (1 June to 30 November) in ERA5, at full resolution, over the whole
basin: detect every storm center, link the centers into tracks, and compute IKE,
peak wind and gale-force radius along each track.

**This section needs NCAR's GLADE file system** (Casper or Derecho), a node with at
least 16 GB of memory, and as many cores as you can get. It reads the ERA5 analyses
that GDEX ([dataset d633000](https://gdex.ucar.edu/datasets/d633000/)) keeps on GLADE.
Set `ERA5_SFC_DIR` to point elsewhere.

| | |
|---|---|
| Grid | ERA5 0.25° latitude-longitude, 0–50°N, 100°W–0°: 80,400 cells, used as an unstructured grid |
| Time | hourly, 4,392 steps |
| Input | 10 m wind components and mean sea-level pressure: about 4 GB actually read from 25 GB of monthly global files, because only the on-disk chunks covering the Atlantic are read |
| Neighborhoods | 2° (detection) and 3° (IKE), each built once in a few seconds |
| Reductions | six per hour, a few milliseconds each per hour of data |

There is no need to coarsen: the reductions take about two minutes on a 12-core
laptop, so reading the data dominates. Expect 5–10 minutes in total. The cell
timings are collected and summarized at the end.

In [ ]:
import glob
import os
import time as clock

import dask

# Use the cores this job was allocated, not every core on the node
n_cores = (
    len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else os.cpu_count()
)
dask.config.set(scheduler="threads", num_workers=n_cores)

era5_dirs = [  # GDEX d633000 surface analyses on GLADE (current and former layout)
    "/glade/campaign/collections/gdex/data/d633000/e5.oper.an.sfc",
    "/glade/campaign/collections/rda/data/ds633.0/e5.oper.an.sfc",
]
era5_dir = os.environ.get("ERA5_SFC_DIR") or next(
    (d for d in era5_dirs if os.path.isdir(d)), None
)
if era5_dir is None:
    raise FileNotFoundError(
        "ERA5 surface analyses not found. This section runs on NCAR's GLADE file "
        "system; set ERA5_SFC_DIR to use a copy elsewhere."
    )

# 10 m u, 10 m v and mean sea-level pressure, one file per variable per month
files = sorted(
    f
    for month in [f"2020{m:02d}" for m in range(6, 12)]
    for code in ["128_165_10u", "128_166_10v", "128_151_msl"]
    for f in glob.glob(f"{era5_dir}/{month}/e5.oper.an.sfc.{code}.ll025sc.*.nc")
)
timings = {}
print(
    f"{len(files)} files, {sum(map(os.path.getsize, files)) / 1e9:.1f} GB; {n_cores} cores"
)

### Read the basin

The files are global and chunked 27 hours × 139 latitudes × 277 longitudes on disk,
so selecting the Atlantic reads only the chunks that overlap it. Matching dask's
time chunks to the on-disk ones avoids reading a chunk twice. `from_structured` turns
the latitude-longitude grid into an unstructured one; each grid point becomes a face.

Only sea-level pressure and wind speed are kept, in memory (`persist`), so the data
is read once and every later step works from memory.

In [ ]:
start = clock.perf_counter()
# Whole Atlantic slices per task: a neighborhood needs every cell at once
era5 = xr.open_mfdataset(
    files, chunks={"time": 27, "latitude": -1, "longitude": -1}, parallel=True
)
era5 = era5[["VAR_10U", "VAR_10V", "MSL"]].sel(
    latitude=slice(50, 0), longitude=slice(260, 359.75)
)
# These files carry units but no CF standard_name, which from_structured looks for
era5.latitude.attrs["standard_name"] = "latitude"
era5.longitude.attrs["standard_name"] = "longitude"

season = ux.UxDataset.from_structured(era5)
atlantic = season.uxgrid
msl = season["MSL"].persist()  # Pa
wind = np.hypot(season["VAR_10U"], season["VAR_10V"]).rename("wind_speed").persist()

timings["read ERA5"] = clock.perf_counter() - start
n_hours = msl.sizes["time"]
print(f"{atlantic.n_face} cells x {n_hours} hours in {timings['read ERA5']:.0f} s")

In [ ]:
start = clock.perf_counter()
nb_detect = atlantic.neighborhood(r=2.0)
nb_storm = atlantic.neighborhood(r=3.0)
timings["build neighborhoods"] = clock.perf_counter() - start
print(nb_detect, nb_storm, sep="\n")

### Detect storm centers

On a latitude-longitude grid the cells shrink toward the pole, so a fixed radius
holds more of them at 40°N than at the equator. Comparing `n_neighbors` with the
median, as in Part 1, would mistake latitude for the domain edge. Instead, compare it
with the number of cells a full disk should hold, its area over the cell's area; at
the edge of the box only part of the disk is there.

A center must be the lowest pressure within 2°, at least 2 hPa below the 2° mean,
and have 15 m/s of wind within 2°. ERA5 underestimates tropical-cyclone winds, so
that last threshold is lower than the 18 m/s that defines a tropical storm. Centers
north of 40°N are dropped to leave out most extratropical lows.

In [ ]:
lat_a, lon_a = atlantic.face_lat.values, atlantic.face_lon.values
full_disk = 2 * np.pi * (1 - np.cos(np.radians(nb_detect.r))) / atlantic.face_areas
allowed = (nb_detect.n_neighbors / full_disk >= 0.95) & (atlantic.face_lat <= 40)

start = clock.perf_counter()
is_min = msl == nb_detect.min(msl)
depth = nb_detect.mean(msl) - msl
peak = nb_detect.max(wind)
centers = is_min & (depth >= 200) & (peak >= 15) & allowed

hit_t, hit_k = np.nonzero(centers.values)
timings["detect centers"] = clock.perf_counter() - start
print(f"{hit_t.size} candidate centers in {n_hours} hours")

### Link centers into tracks

Each open track takes the nearest center within 1° per hour; a track may go
undetected for up to 3 hours, and only tracks lasting at least a day are kept. This
is the simplest version of what dedicated trackers such as TempestExtremes do.

In [ ]:
def great_circle(lat1, lon1, lat2, lon2):
    # Great-circle distance in degrees (haversine)
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = (
        np.sin((p2 - p1) / 2) ** 2
        + np.cos(p1) * np.cos(p2) * np.sin(np.radians(lon2 - lon1) / 2) ** 2
    )
    return np.degrees(2 * np.arcsin(np.sqrt(a)))


MAX_STEP, MAX_GAP, MIN_HOURS = 1.0, 3, 24  # degrees per hour, hours, hours

by_hour = np.split(hit_k, np.searchsorted(hit_t, np.arange(1, n_hours)))
finished, active = [], []
for t, cells in enumerate(by_hour):
    unused, still_active = list(cells), []
    for track in sorted(active, key=len, reverse=True):  # longest tracks pick first
        last_t, last_k = track[-1]
        if t - last_t > MAX_GAP:
            finished.append(track)
            continue
        if unused:
            dist = great_circle(
                lat_a[last_k], lon_a[last_k], lat_a[unused], lon_a[unused]
            )
            j = int(np.argmin(dist))
            if dist[j] <= MAX_STEP * (t - last_t):
                track.append((t, unused.pop(j)))
        still_active.append(track)
    active = still_active + [[(t, k)] for k in unused]

tracks = [
    np.array(tr) for tr in finished + active if tr[-1][0] - tr[0][0] + 1 >= MIN_HOURS
]
print(f"{len(tracks)} tracks lasting at least {MIN_HOURS} h")

### IKE, peak wind and gale-force radius along every track

As in Part 2, but for every hour of every track in one pass. Each reduction covers
the whole basin at every hour, and only the values at track positions are kept.

In [ ]:
start = clock.perf_counter()
area = atlantic.face_areas * R_EARTH**2  # m²
gale = wind >= 18  # m/s
ke = (0.5 * 1.0 * wind**2).where(gale, 0) * area  # J per 1 m deep layer

fix_t = np.concatenate([tr[:, 0] for tr in tracks])
fix_k = np.concatenate([tr[:, 1] for tr in tracks])
pick = dict(
    time=xr.DataArray(fix_t, dims="fix"), n_face=xr.DataArray(fix_k, dims="fix")
)
ike_fix, vmax_fix, gale_fix = dask.compute(
    xr.DataArray(nb_storm.sum(ke)).isel(pick),
    xr.DataArray(nb_storm.max(wind)).isel(pick),
    xr.DataArray(nb_storm.sum(gale * area)).isel(pick),
)
timings["IKE along tracks"] = clock.perf_counter() - start

fixes = pd.DataFrame(
    {
        "track": np.repeat(np.arange(len(tracks)), [len(tr) for tr in tracks]),
        "time": msl.time.values[fix_t],
        "lat": lat_a[fix_k],
        "lon": lon_a[fix_k],
        "ike_tj": ike_fix.values / 1e12,
        "vmax": vmax_fix.values,
        "gale_radius_km": np.sqrt(gale_fix.values / np.pi) / 1e3,
    }
)
fixes

### Name the tracks

The [IBTrACS](https://www.ncei.noaa.gov/products/international-best-track-archive)
best tracks (downloaded from NOAA, about 60 MB) give each detected track a name: the
storm whose best-track position is within 2.5° at the same times. Tracks without a
match are other lows, such as monsoon troughs or systems that never became named
storms. Without internet access the tracks stay numbered.

In [ ]:
IBTRACS = os.environ.get(
    "IBTRACS_CSV",
    "https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/ibtracs.NA.list.v04r01.csv",
)
try:
    best = pd.read_csv(
        IBTRACS,
        skiprows=[1],  # units row
        usecols=[
            "SEASON",
            "BASIN",
            "NAME",
            "ISO_TIME",
            "NATURE",
            "LAT",
            "LON",
            "USA_WIND",
        ],
        keep_default_na=False,  # otherwise the North Atlantic basin code "NA" reads as missing
        low_memory=False,
    )
    best = best[(best.SEASON.astype(str) == "2020") & (best.BASIN == "NA")]
    best = best.assign(
        time=pd.to_datetime(best.ISO_TIME),
        LAT=pd.to_numeric(best.LAT),
        LON=pd.to_numeric(best.LON),
        USA_WIND=pd.to_numeric(best.USA_WIND, errors="coerce"),
    )
except Exception as err:  # no internet access, for example
    best = None
    print(f"IBTrACS unavailable ({type(err).__name__}); tracks stay numbered")


def name_of(track):
    if best is None:
        return f"track {track.name}"
    pairs = track.merge(best, on="time")
    pairs = pairs[great_circle(pairs.lat, pairs.lon, pairs.LAT, pairs.LON) <= 2.5]
    return pairs.NAME.mode().iloc[0] if len(pairs) else None


names = fixes.groupby("track")[["time", "lat", "lon"]].apply(name_of)
fixes["name"] = fixes.track.map(names)
if best is not None:
    print(
        f"{names.notna().sum()} of {len(names)} tracks matched; "
        f"{fixes.name.nunique()} of {best.NAME.nunique()} named storms found"
    )

### The season by storm

Accumulated IKE is the time integral of IKE over each storm's life in ERA5. ACE is
computed both from ERA5's peak wind and from the observed best-track wind, at the
6-hourly fixes ACE is defined on (tropical and subtropical stages only, winds of at
least 35 kt). Expect ERA5's ACE to fall well below the observed: a 0.25° reanalysis
underestimates peak winds much more than it underestimates the broad wind field that
IKE integrates. Compare how the two rank the storms.

In [ ]:
def ace(knots):
    return 1e-4 * (knots.where(knots >= 35, 0) ** 2).sum()


named = fixes.dropna(subset=["name"])
synoptic = named[named.time.dt.hour % 6 == 0]
storms = pd.DataFrame(
    {
        "first fix": named.groupby("name").time.min().dt.strftime("%d %b"),
        "peak IKE (TJ)": named.groupby("name").ike_tj.max(),
        "accumulated IKE (TJ day)": named.groupby("name").ike_tj.sum() / 24,
        "ERA5 ACE": synoptic.groupby("name").vmax.apply(lambda v: ace(v * 1.94384)),
    }
)
if best is not None:
    observed = best[
        best.time.dt.hour.isin([0, 6, 12, 18]) & best.NATURE.isin(["TS", "SS"])
    ]
    storms["observed ACE"] = observed.groupby("NAME").USA_WIND.apply(ace)
storms = storms.sort_values("accumulated IKE (TJ day)", ascending=False).round(1)
storms

In [ ]:
top = storms.index[:8]
bars = dict(width=420, height=320, xrotation=60, tools=["hover"])
(
    hv.Overlay(
        [
            hv.Curve((g.time.values, g.ike_tj.values), "time", "IKE (TJ)", label=name)
            for name, g in named[named.name.isin(top)].groupby("name")
        ]
    ).opts(
        width=860,
        height=300,
        legend_position="right",
        title="IKE, eight largest storms",
    )
    + hv.Bars(storms["accumulated IKE (TJ day)"].reset_index(), "name").opts(
        **bars, title="Accumulated IKE"
    )
    + hv.Bars(
        storms[["observed ACE" if best is not None else "ERA5 ACE"]].reset_index(),
        "name",
    ).opts(**bars, title="ACE")
).cols(1)

### The season's wind footprint

The Part 2 trick at basin scale: mark every track position, dilate the marks by 3°
with a neighborhood max, and take the strongest wind under them over the season.

In [ ]:
start = clock.perf_counter()
marks = np.zeros(wind.shape, dtype="float32")
marks[fix_t, fix_k] = 1
marks = ux.UxDataArray(marks, dims=wind.dims, uxgrid=atlantic).chunk(time=27)
near_track = (nb_storm.max(marks) > 0).persist()
season_footprint = (
    wind.where(near_track, 0)
    .max("time")
    .where(near_track.any("time"))  # blank away from every track
    .rename("max_wind")
    .compute()
)
timings["footprint"] = clock.perf_counter() - start

paths = [
    hv.Path([np.column_stack([g.lon, g.lat])]).opts(
        color="white" if g.name.notna().all() else "gray",
        line_dash="solid" if g.name.notna().all() else "dashed",
        line_width=1.5,
    )
    for _, g in fixes.groupby("track")
]
starts = fixes.dropna(subset=["name"]).groupby("name")[["lon", "lat"]].first()
season_footprint.plot.polygons(
    cmap="inferno",
    clabel="Max 10 m wind (m/s)",
    title="2020 Atlantic season in ERA5 (gray dashed: unnamed)",
    width=900,
    height=500,
    geo=True,
    coastline=True,
) * hv.Overlay(paths) * hv.Labels(
    (starts.lon, starts.lat, starts.index.values), ["x", "y"], "name"
).opts(text_color="white", text_font_size="8pt", yoffset=0.6)

In [ ]:
timings = pd.Series(timings, name="seconds").round(1)
timings["total"] = timings.sum()
timings.to_frame()

## Caveats

- **The radius is a great-circle distance in degrees**: 1° ≈ 111 km at any latitude,
  unlike a box in longitude and latitude.
- **Disk-edge resolution**: a cell counts entirely if its center is inside the disk,
  so a disk integral is accurate to about one cell along its edge. That is small when
  `r` spans many cells, as here (3° on a ~0.25° grid).
- **Regional domains**: near the boundary neighborhoods are truncated; mask with
  `n_neighbors` as in Part 1.
- **Nearby storms**: two storms closer than `r` count each other's winds.
- **Model winds are not observed winds**: IKE and ACE are defined on 1-minute
  sustained 10 m winds, and a ~25 km model underestimates peak winds.

## See Also

- [Neighborhood Filter](neighborhood-filter.ipynb): the full set of reductions,
  node- and edge-centered data, and custom reductions
- {py:meth}`uxarray.Grid.neighborhood`
- {py:meth}`uxarray.UxDataArray.azimuthal_mean`: radial profiles around a point,
  for example the radius of maximum wind